# CNN on Fashion MNIST

Convolutional neural networks exploit spatial structure in images. This notebook trains a small CNN on Fashion MNIST and compares its accuracy with a flat baseline.

## Imports and data

We download Fashion MNIST — 70 000 grayscale images of clothing items (T-shirts, trousers, sneakers, etc.) with the same 28×28 format as MNIST. The dataset is wrapped in `DataLoader` objects for batched training and evaluation.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = Path("../../data/raw")

transform = transforms.ToTensor()
train_ds = datasets.FashionMNIST(data_dir, train=True, download=True, transform=transform)
test_ds = datasets.FashionMNIST(data_dir, train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256)

class_names = train_ds.classes
print(f"Train: {len(train_ds)} | Test: {len(test_ds)} | Classes: {class_names}")

## Sample images

We visualise a few training examples with their class names to see the kind of items the model will learn to classify.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(12, 1.5))
for i, ax in enumerate(axes):
    img, label = train_ds[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(class_names[label], fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Define model

We build a small CNN with two convolutional layers (16 and 32 filters), each followed by ReLU and max-pooling. The feature maps are then flattened and passed through a fully-connected classifier. Unlike the MLP, the CNN preserves spatial structure and learns local patterns like edges and textures.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model

## Train and evaluate

The training loop runs for 5 epochs. Each epoch processes all training batches (forward pass → loss → backpropagation → weight update), then evaluates accuracy on the test set.

In [ ]:
history = {"epoch": [], "loss": [], "accuracy": []}

for epoch in range(1, 6):
    model.train()
    total_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = loss_fn(model(images), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    model.eval()
    correct = sum(
        (model(img.to(device)).argmax(1) == lab.to(device)).sum().item()
        for img, lab in test_loader
    )
    acc = correct / len(test_ds)
    avg_loss = total_loss / len(train_loader)

    history["epoch"].append(epoch)
    history["loss"].append(avg_loss)
    history["accuracy"].append(acc)
    print(f"Epoch {epoch}: loss={avg_loss:.4f}, accuracy={acc:.3f}")

## Training curves

We plot loss and accuracy over epochs to verify the model is converging and not overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history["epoch"], history["loss"], marker="o")
ax1.set(xlabel="Epoch", ylabel="Loss", title="Training loss")
ax2.plot(history["epoch"], history["accuracy"], marker="o", color="green")
ax2.set(xlabel="Epoch", ylabel="Accuracy", title="Test accuracy")
plt.tight_layout()
plt.show()

## Practical note

CNNs exploit spatial locality through convolutions and pooling layers, making them far more parameter-efficient than MLPs for image data. This small two-layer CNN already outperforms a much larger MLP on the same data.